# Next Word Prediction with an LSTM (Shakespeare's *Hamlet*)

This notebook trains the model used by the Streamlit app (`app.py`).

Run every cell from top to bottom. It will:

1. Load the text of *Hamlet* (`hamlet.txt`)
2. Turn each line into next-word training examples
3. Train an LSTM on the **whole play** until it has memorised it
4. Save `next_word_lstm.h5` and `tokenizer.pickle`
5. Try a few predictions

Then start the app with `streamlit run app.py`.

Training takes roughly 10-15 minutes on a laptop CPU.

## 1. Setup

In [1]:
import os
import pickle
import platform
from collections import Counter, defaultdict

import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

tf.keras.utils.set_random_seed(42)

# On Apple Silicon the Metal GPU is slower than the CPU for a small LSTM like this one.
if platform.system() == "Darwin":
    tf.config.set_visible_devices([], "GPU")

print("TensorFlow", tf.__version__)

TensorFlow 2.15.0


## 2. Settings

Training stops when **both** are true:

- the model predicts `CHECK_WORD` as the next word after `CHECK_TEXT`
- training accuracy is close to the best accuracy that is possible on this text (explained in section 4)

In [2]:
DATA_FILE = "hamlet.txt"
MODEL_FILE = "next_word_lstm.h5"
TOKENIZER_FILE = "tokenizer.pickle"

CHECK_TEXT = "Hor. My Lord, I came to see your Fathers"
CHECK_WORD = "funerall"

MAX_EPOCHS = 300        # safety limit; training normally stops earlier
BATCH_SIZE = 64
LEARNING_RATE = 0.003
ACCURACY_GOAL = 0.97    # fraction of the best possible accuracy to reach

## 3. Load the data

`hamlet.txt` is included in the repository. If it is missing, it is downloaded from the NLTK Gutenberg corpus.

In [3]:
if not os.path.exists(DATA_FILE):
    import nltk
    nltk.download("gutenberg")
    from nltk.corpus import gutenberg
    with open(DATA_FILE, "w") as file:
        file.write(gutenberg.raw("shakespeare-hamlet.txt"))

with open(DATA_FILE) as file:
    text = file.read().lower()

print(f"{len(text):,} characters, {len(text.splitlines()):,} lines")
print(text[:300])

162,881 characters, 4,922 lines
[the tragedie of hamlet by william shakespeare 1599]


actus primus. scoena prima.

enter barnardo and francisco two centinels.

  barnardo. who's there?
  fran. nay answer me: stand & vnfold
your selfe

   bar. long liue the king

   fran. barnardo?
  bar. he

   fran. you come most carefully vpon 


## 4. Preprocessing

**Tokenise:** give every distinct word a number.

In [4]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

print("Vocabulary size:", total_words)
print("Most common words:", list(tokenizer.word_index.items())[:10])

Vocabulary size: 4818
Most common words: [('the', 1), ('and', 2), ('to', 3), ('of', 4), ('i', 5), ('you', 6), ('a', 7), ('my', 8), ('it', 9), ('in', 10)]


**Build training examples:** every line is expanded into all of its prefixes. For the line
`my lord i came` we get `my lord`, `my lord i`, `my lord i came`. The last word of each prefix is the
word to predict and everything before it is the input.

Lines are handled one at a time, so the model never sees a phrase that continues across a line break.

In [5]:
input_sequences = []
for line in text.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[: i + 1])

max_sequence_len = max(len(seq) for seq in input_sequences)
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding="pre"))

x, y = input_sequences[:, :-1], input_sequences[:, -1]

print("Training examples:", len(y))
print("Longest sequence:", max_sequence_len)
print("x shape:", x.shape, " y shape:", y.shape)

Training examples: 25732
Longest sequence: 14
x shape: (25732, 13)  y shape: (25732,)


**Best possible accuracy:** the same input can be followed by different words in different places
(a line starting with `ham` continues in hundreds of ways), and the model can only give one answer per input.
So 100% accuracy is impossible. The ceiling is what you would get by always answering with the most
common next word for each input.

In [6]:
next_words_by_input = defaultdict(Counter)
for inputs, next_word in zip(map(bytes, x), y):
    next_words_by_input[inputs][next_word] += 1

accuracy_ceiling = sum(c.most_common(1)[0][1] for c in next_words_by_input.values()) / len(y)
target_accuracy = ACCURACY_GOAL * accuracy_ceiling

print(f"Best possible accuracy: {accuracy_ceiling:.4f}")
print(f"Training target:        {target_accuracy:.4f}")

Best possible accuracy: 0.8525
Training target:        0.8269


## 5. Build the model

Embedding -> LSTM(150) -> LSTM(100) -> softmax over the vocabulary.

There is no train/test split and no dropout: the goal is to memorise the play, not to generalise to new text.

In [7]:
model = Sequential([
    Embedding(total_words, 100, input_length=max_sequence_len - 1),
    LSTM(150, return_sequences=True),
    LSTM(100),
    Dense(total_words, activation="softmax"),
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE),
    metrics=["accuracy"],
)
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 13, 100)           481800    
                                                                 
 lstm (LSTM)                 (None, 13, 150)           150600    
                                                                 
 lstm_1 (LSTM)               (None, 100)               100400    
                                                                 
 dense (Dense)               (None, 4818)              486618    
                                                                 
Total params: 1219418 (4.65 MB)
Trainable params: 1219418 (4.65 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


## 6. Train

After every epoch the callback prints the loss, the accuracy and the model's current guess for `CHECK_TEXT`,
and stops training once the stopping rule from section 2 is met.

In [8]:
check_x = pad_sequences(
    [tokenizer.texts_to_sequences([CHECK_TEXT])[0][-(max_sequence_len - 1):]],
    maxlen=max_sequence_len - 1,
    padding="pre",
)


class StopWhenMemorized(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        probabilities = self.model.predict(check_x, verbose=0)[0]
        guess = tokenizer.index_word.get(int(np.argmax(probabilities)))
        accuracy = logs["accuracy"]
        print(
            f"epoch {epoch + 1:3d}  loss={logs['loss']:.4f}  accuracy={accuracy:.4f}  "
            f"check -> {guess} ({probabilities.max():.1%})",
            flush=True,
        )
        if guess == CHECK_WORD and accuracy >= target_accuracy:
            print("Stopping: check word predicted and accuracy target reached.")
            self.model.stop_training = True


history = model.fit(
    x,
    y,
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=0,
    callbacks=[
        StopWhenMemorized(),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="loss", factor=0.5, patience=4, min_lr=1e-4),
    ],
)

epoch   1  loss=7.1010  accuracy=0.0298  check -> the (3.2%)
epoch   2  loss=6.6730  accuracy=0.0333  check -> the (3.1%)
epoch   3  loss=6.6285  accuracy=0.0343  check -> the (1.0%)
epoch   4  loss=6.4497  accuracy=0.0450  check -> lord (0.7%)
epoch   5  loss=6.1766  accuracy=0.0603  check -> lord (2.0%)
epoch   6  loss=5.9321  accuracy=0.0769  check -> king (4.3%)
epoch   7  loss=5.7320  accuracy=0.0869  check -> king (3.4%)
epoch   8  loss=5.5407  accuracy=0.0983  check -> king (3.2%)
epoch   9  loss=5.3577  accuracy=0.1086  check -> death (4.1%)
epoch  10  loss=5.1822  accuracy=0.1159  check -> death (4.9%)
epoch  11  loss=5.0121  accuracy=0.1247  check -> death (5.9%)
epoch  12  loss=4.8354  accuracy=0.1332  check -> death (6.8%)
epoch  13  loss=4.6704  accuracy=0.1447  check -> death (15.1%)
epoch  14  loss=4.4986  accuracy=0.1565  check -> death (10.9%)
epoch  15  loss=4.3306  accuracy=0.1678  check -> death (13.6%)
epoch  16  loss=4.1678  accuracy=0.1845  check -> death (15.0%)

## 7. Save the model and tokenizer

In [9]:
model.save(MODEL_FILE)
with open(TOKENIZER_FILE, "wb") as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("Epochs trained:", len(history.history["loss"]))
print(f"Final accuracy: {history.history['accuracy'][-1]:.4f}")
print("Saved", MODEL_FILE, "and", TOKENIZER_FILE)

Epochs trained: 109
Final accuracy: 0.8352
Saved next_word_lstm.h5 and tokenizer.pickle


## 8. Try it

Each input below is the start of a line from the play; the word in the comment is what the play says next.

In [10]:
def predict_next_words(model, tokenizer, text, top_k=3):
    sequence_len = model.input_shape[1]
    token_list = tokenizer.texts_to_sequences([text])[0][-sequence_len:]
    token_list = pad_sequences([token_list], maxlen=sequence_len, padding="pre")
    probabilities = model.predict(token_list, verbose=0)[0]
    best = np.argsort(probabilities)[::-1][:top_k]
    return [(tokenizer.index_word[i], float(probabilities[i])) for i in best]


examples = [
    "Hor. My Lord, I came to see your Fathers",   # Funerall
    "To be, or not to be, that is the",           # Question
    "Ham. I Madam, it is",                        # common
    "Barnardo. Who's",                            # there
]

for example in examples:
    guesses = ", ".join(f"{word} ({p:.1%})" for word, p in predict_next_words(model, tokenizer, example))
    print(f"{example!r}\n   -> {guesses}\n")

'Hor. My Lord, I came to see your Fathers'
   -> funerall (99.8%), death (0.2%), beleeue (0.0%)

'To be, or not to be, that is the'
   -> question (92.1%), poyson (2.8%), carpenter (0.8%)

'Ham. I Madam, it is'
   -> common (93.6%), most (1.4%), no (0.9%)

"Barnardo. Who's"
   -> there (98.6%), a (1.0%), good (0.1%)



## 9. Run the app

```
streamlit run app.py
```

The app loads `next_word_lstm.h5` and `tokenizer.pickle` from this folder.